# LeetCode 774 — Minimize Max Distance to Gas Station bằng Python

- Link: https://leetcode.com/problems/minimize-max-distance-to-gas-station/
- Độ khó: **Hard**
- Chủ đề: Tìm kiếm nhị phân trên đáp án (số thực), Tham lam, Heap (hàng đợi ưu tiên), Toán (làm tròn lên/xuống)

> **Ghi chú (bài Premium):** đề bài dưới đây được viết lại theo hiểu biết về bài toán (ví dụ, ràng buộc, sai số chấp nhận `10^-6`). Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại phát biểu và chữ ký hàm chính thức.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minmaxGasDist(self, stations: List[int], k: int) -> float:
```

## 1. Giới thiệu & mục tiêu học

Trên một con đường thẳng có sẵn các trạm xăng ở những vị trí nguyên. Ta được xây thêm đúng `k` trạm mới ở **vị trí bất kỳ** (kể cả số thực). Mục tiêu: làm cho **khoảng cách lớn nhất giữa hai trạm liền kề** càng nhỏ càng tốt.

Đây là bài kinh điển của mẫu **"cực tiểu hoá giá trị lớn nhất" → tìm kiếm nhị phân trên đáp án**, và đặc biệt ở chỗ đáp án là **số thực**. Bài cũng có lời giải tham lam bằng heap rất tự nhiên — nhưng chậm khi `k` lớn — và một mẹo "chia theo tỉ lệ trước" giúp heap chạy nhanh.

Sau notebook này bạn sẽ:

1. Nhận ra rằng chỉ có **các khoảng trống** (gap) giữa các trạm là quan trọng, và chia một khoảng thành `p` phần đều là tối ưu.
2. Viết lời giải tham lam "mỗi lần thêm một trạm vào khoảng đang dài nhất" (quét tuyến tính, rồi heap).
3. Viết **tìm kiếm nhị phân trên số thực** với hàm kiểm tra `cần bao nhiêu trạm để mọi đoạn ≤ D`.
4. Kết hợp **phân bổ theo tỉ lệ + heap** để có lời giải chính xác `O(n log n)`.
5. Biết cách so sánh số thực với sai số và chọn số vòng lặp nhị phân.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng số nguyên `stations` **tăng ngặt** — vị trí các trạm xăng trên trục số — và số nguyên `k`. Ta thêm đúng `k` trạm mới ở vị trí bất kỳ (không nhất thiết nguyên). Gọi **penalty** là khoảng cách lớn nhất giữa hai trạm liền kề (sau khi thêm). Trả về penalty **nhỏ nhất** có thể. Đáp án sai lệch không quá `10^-6` so với đáp án đúng được chấp nhận.

**Ví dụ 1**

```text
Input: stations = [1,2,3,4,5,6,7,8,9,10], k = 9
Output: 0.50000
Giải thích: đặt một trạm vào giữa mỗi khoảng (9 khoảng, mỗi khoảng dài 1) -> mọi đoạn dài 0.5.
```

**Ví dụ 2**

```text
Input: stations = [23,24,36,39,46,56,57,65,84,98], k = 1
Output: 14.00000
Giải thích: khoảng dài nhất 65->84 (19) được chia đôi còn 9.5; khoảng 84->98 (14) trở thành dài nhất.
```

**Ràng buộc** (theo hiểu biết)

- `10 <= stations.length <= 2000`
- `0 <= stations[i] <= 10^8`
- `stations` tăng ngặt
- `1 <= k <= 10^6`

In [1]:
from typing import List
import math

EXAMPLES = [
    (list(range(1, 11)), 9, 0.5),
    ([23, 24, 36, 39, 46, 56, 57, 65, 84, 98], 1, 14.0),
]

def close(a, b, eps=1e-6):
    return abs(a - b) <= eps

gaps = [b - a for a, b in zip(EXAMPLES[1][0], EXAMPLES[1][0][1:])]
print("các khoảng trống của ví dụ 2:", gaps)

các khoảng trống của ví dụ 2: [1, 12, 3, 7, 10, 1, 8, 19, 14]


## 3. Kiến thức nền

### 3.1. Khoảng trống giữa các phần tử liền kề

`zip(a, a[1:])` cho các cặp liền kề → danh sách gap. Cả bài chỉ phụ thuộc vào danh sách này.

In [2]:
st = [1, 4, 10]
print([b - a for a, b in zip(st, st[1:])])

[3, 6]


### 3.2. Chia một khoảng thành `p` phần: phần lớn nhất nhỏ nhất là chia đều

Chèn `p - 1` trạm vào khoảng dài `g` → `p` đoạn. Đoạn lớn nhất nhỏ nhất khi chia đều: `g / p`. Ngược lại, muốn mọi đoạn `≤ D` thì cần ít nhất `p = ⌈g / D⌉` đoạn, tức `⌈g / D⌉ − 1` trạm.

In [3]:
g = 19
for D in [9.5, 10, 6.4, 19, 20]:
    print(f"D = {D:>5}: cần {math.ceil(g / D) - 1} trạm (chia thành {math.ceil(g / D)} đoạn)")

D =   9.5: cần 1 trạm (chia thành 2 đoạn)
D =    10: cần 1 trạm (chia thành 2 đoạn)
D =   6.4: cần 2 trạm (chia thành 3 đoạn)
D =    19: cần 0 trạm (chia thành 1 đoạn)
D =    20: cần 0 trạm (chia thành 1 đoạn)


### 3.3. Tìm kiếm nhị phân trên số thực

Với hàm **đơn điệu** `ok(D)` (sai với `D` nhỏ, đúng với `D` lớn), ta thu hẹp `[lo, hi]`: `mid = (lo + hi) / 2`, nếu `ok(mid)` thì `hi = mid`, ngược lại `lo = mid`. Dừng khi `hi - lo < eps` hoặc sau số vòng cố định (mỗi vòng giảm một nửa: 100 vòng giảm `2^100` lần — dư dả).

Ví dụ: tính `√2` bằng nhị phân.

In [4]:
lo, hi = 0.0, 2.0
for _ in range(60):
    mid = (lo + hi) / 2
    if mid * mid >= 2:
        hi = mid
    else:
        lo = mid
print(hi, math.sqrt(2))

1.4142135623730951 1.4142135623730951


### 3.4. Heap với "giá trị âm" để làm max-heap

`heapq` là min-heap. Để lấy phần tử lớn nhất, đẩy `-giá_trị`. Ở bài này ta đẩy `(-độ_dài_đoạn_hiện_tại, chỉ_số_khoảng)`.

In [5]:
import heapq
h = []
for v in [3, 9, 1, 7]:
    heapq.heappush(h, -v)
print([-heapq.heappop(h) for _ in range(4)])

[9, 7, 3, 1]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Chỉ có gap quan trọng

Trạm mới đặt ngoài đoạn `[stations[0], stations[-1]]` vô ích. Bên trong, mỗi trạm mới rơi vào một gap. Gọi `p_i` là số **đoạn** mà gap `i` bị chia thành (`p_i − 1` trạm). Penalty = `max_i g_i / p_i` (chia đều). Bài toán:

$$\min \max_i \frac{g_i}{p_i} \quad \text{với } p_i \ge 1 \text{ nguyên}, \sum_i (p_i - 1) = k.$$

### 4.2. Tính tay ví dụ 2 (`k = 1`)

Gaps: `1, 12, 3, 7, 10, 1, 8, 19, 14`. Khoảng lớn nhất là 19. Đặt trạm duy nhất vào đó: `19 / 2 = 9.5`. Max mới = `max(9.5, 14, 12, ...)` = **14**. Đặt vào chỗ khác thì 19 vẫn còn → tệ hơn. Đáp án 14.

### 4.3. Tính tay một ví dụ nhỏ: gaps `[10, 6]`, `k = 3`

- Thêm trạm 1: khoảng dài nhất 10 → `p = [2, 1]`, các đoạn `5, 6`.
- Thêm trạm 2: dài nhất là 6 → `p = [2, 2]`, đoạn `5, 3`.
- Thêm trạm 3: dài nhất 5 → `p = [3, 2]`, đoạn `3.33, 3`.
- Đáp án `10/3 ≈ 3.333`.

### 4.4. Đơn điệu theo D

Gọi `need(D) = Σ (⌈g_i / D⌉ − 1)` — số trạm tối thiểu để mọi đoạn `≤ D`. `D` càng lớn, `need(D)` càng nhỏ (không tăng). Đáp án là `D` nhỏ nhất với `need(D) ≤ k` → tìm kiếm nhị phân.

In [6]:
def need(gaps, D):
    return sum(math.ceil(g / D) - 1 for g in gaps)

g2 = [1, 12, 3, 7, 10, 1, 8, 19, 14]
for D in [20, 14, 13.99, 10, 9.5, 5]:
    print(f"need({D}) = {need(g2, D)}")

need(20) = 0
need(14) = 1
need(13.99) = 2
need(10) = 3
need(9.5) = 4
need(5) = 10


## 5. Cách 1 — Brute Force: tham lam từng trạm, quét tìm khoảng dài nhất

**Ý tưởng.** Giữ `p_i` (số đoạn của gap `i`, ban đầu 1). Lặp `k` lần: tìm gap có `g_i / p_i` lớn nhất bằng **quét tuyến tính**, tăng `p_i` lên 1. Cuối cùng penalty = `max g_i / p_i`.

(Ta sẽ chứng minh tham lam này tối ưu ở mục 10.)

In [7]:
def greedy_scan(stations, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    p = [1] * len(gaps)
    for _ in range(k):
        best = max(range(len(gaps)), key=lambda i: gaps[i] / p[i])
        p[best] += 1
    return max(g / c for g, c in zip(gaps, p))

for st, k, exp in EXAMPLES:
    print(greedy_scan(st, k), exp)
print(greedy_scan([0, 10, 16], 3))

0.5 0.5
14.0 14.0
3.3333333333333335


**Độ phức tạp.** `O(k · n)`: với `k = 10^6`, `n = 2000` là `2·10^9` — quá chậm.

**Nút thắt.** (1) Mỗi lần tìm max tốn `O(n)` → dùng **heap** được `O(log n)`. (2) Dù vậy vẫn phải lặp `k` lần (tới `10^6`). Để thoát khỏi phụ thuộc vào `k`, ta **đoán đáp án** `D` và kiểm tra bằng công thức đóng `need(D)` — tìm kiếm nhị phân.

## 6. Bước đột phá

**Insight 1 — Đảo bài toán (tối ưu → quyết định).** Thay vì hỏi "penalty nhỏ nhất là bao nhiêu?", hỏi "với penalty `D`, có làm được bằng `≤ k` trạm không?". Câu hỏi quyết định trả lời được ngay: `need(D) = Σ (⌈g_i / D⌉ − 1) ≤ k`.

**Insight 2 — Tính đơn điệu.** Làm được với `D` thì làm được với mọi `D' > D`. Nên tập các `D` làm được là một nửa đường thẳng `[D*, +∞)` → tìm kiếm nhị phân trên `D` (số thực).

**Insight 3 — Cận.** `lo = 0` (không làm được), `hi = max(g_i)` (làm được với 0 trạm). 100 vòng nhị phân cho sai số `10^8 / 2^100` — cực nhỏ; hoặc dừng khi `hi − lo < 10^-6`.

**Insight 4 — Tham lam với điểm xuất phát tốt.** Với `p_i = 1 + ⌊g_i · k / L⌋` (`L = tổng gap`), ta dùng tối đa `k` trạm và mọi đoạn `< L / k`. Có thể chứng minh phân bổ này **không vượt** phân bổ tối ưu ở từng gap → chỉ còn tối đa `n` trạm cần phân bổ thêm bằng heap.

**Quy tắc nếu… thì…**

- Nếu `need(mid) <= k` → `mid` làm được → `hi = mid`.
- Ngược lại → `lo = mid`.
- Nếu cần lời giải chính xác tuyệt đối (không sai số nhị phân) → tham lam + heap với phân bổ tỉ lệ.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tham lam với heap

**Giải thích.** Heap chứa `(-g_i / p_i, i)`. Lặp `k` lần: lấy gap có đoạn dài nhất, tăng `p_i`, đẩy lại với độ dài mới. Kết quả: đỉnh heap.

In [8]:
def greedy_heap(stations, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    p = [1] * len(gaps)
    heap = [(-g, i) for i, g in enumerate(gaps)]
    heapq.heapify(heap)
    for _ in range(k):
        _, i = heapq.heappop(heap)
        p[i] += 1
        heapq.heappush(heap, (-gaps[i] / p[i], i))
    return -heap[0][0]

for st, k, exp in EXAMPLES:
    print(greedy_heap(st, k), exp)

0.5 0.5
14 14.0


**Code nộp LeetCode (Cách 2)** — đúng, nhưng với `k = 10^6` Python có thể **TLE** (khoảng `10^6 · log 2000` thao tác heap).

In [9]:
class SolutionHeap:
    def minmaxGasDist(self, stations: List[int], k: int) -> float:
        import heapq
        gaps = [b - a for a, b in zip(stations, stations[1:])]
        cnt = [1] * len(gaps)
        heap = [(-g, i) for i, g in enumerate(gaps)]
        heapq.heapify(heap)
        for _ in range(k):
            _, i = heapq.heappop(heap)
            cnt[i] += 1
            heapq.heappush(heap, (-gaps[i] / cnt[i], i))
        return -heap[0][0]

print([SolutionHeap().minmaxGasDist(st, k) for st, k, _ in EXAMPLES])

[0.5, 14]


**Độ phức tạp.** `O(n + k log n)` thời gian, `O(n)` bộ nhớ.

### 7.2. Cách 3 — Tìm kiếm nhị phân trên đáp án (số thực)

**Giải thích.** `lo = 0`, `hi = max gap`. Lặp ~100 lần (hoặc tới khi `hi − lo < 1e-6`): `mid = (lo+hi)/2`; tính `need(mid)`; nếu `<= k` thì `hi = mid` ngược lại `lo = mid`. Trả về `hi` (luôn là giá trị làm được).

Trong code ta hay dùng `int(g / D)` thay cho `⌈g / D⌉ − 1`. Hai biểu thức chỉ khác nhau khi `g / D` là số nguyên (khi đó `int` lớn hơn 1) — điều này chỉ làm điều kiện **chặt hơn** đúng tại một điểm, không ảnh hưởng tới giới hạn khi nhị phân trên số thực.

In [10]:
def binary_search(stations, k, iters=100):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    lo, hi = 0.0, float(max(gaps))
    for _ in range(iters):
        mid = (lo + hi) / 2
        if sum(math.ceil(g / mid) - 1 for g in gaps) <= k:
            hi = mid
        else:
            lo = mid
    return hi

for st, k, exp in EXAMPLES:
    print(binary_search(st, k), exp)

0.5 0.5
14.0 14.0


**Code nộp LeetCode (Cách 3).**

In [11]:
class SolutionBinary:
    def minmaxGasDist(self, stations: List[int], k: int) -> float:
        gaps = [b - a for a, b in zip(stations, stations[1:])]
        lo, hi = 0.0, float(max(gaps))
        while hi - lo > 1e-6:
            mid = (lo + hi) / 2
            if sum(int(g / mid) for g in gaps) <= k:   # số trạm cần để mọi đoạn <= mid
                hi = mid
            else:
                lo = mid
        return hi

print([SolutionBinary().minmaxGasDist(st, k) for st, k, _ in EXAMPLES])

[0.5000009536743164, 14.000000387430191]


**Độ phức tạp.** Số vòng `log2(max_gap / eps)` ≈ `log2(10^8 / 10^-6)` ≈ 47; mỗi vòng `O(n)` → `O(n · 47)`. Bộ nhớ `O(n)`.

### 7.3. Cách 4 — Phân bổ theo tỉ lệ + heap (chính xác, `O(n log n)`)

**Giải thích.** Đặt `L = Σ g_i`. Khởi tạo `p_i = 1 + ⌊g_i · k / L⌋`.

- Số trạm đã dùng: `Σ ⌊g_i k / L⌋ ≤ Σ g_i k / L = k`. Còn lại `r = k − Σ⌊…⌋`; vì mỗi phần làm tròn mất `< 1`, `r < n`.
- Mỗi đoạn hiện tại `g_i / p_i < g_i / (g_i k / L) = L / k`. Vậy đáp án tối ưu `D* < L / k`.
- Phân bổ tối ưu cần `p_i* = ⌈g_i / D*⌉ > g_i k / L ≥ ⌊g_i k / L⌋` → `p_i* ≥ p_i`. Phân bổ ban đầu **không vượt** tối ưu ở gap nào.

Từ trạng thái này, chạy tham lam heap cho `r < n` trạm còn lại → kết quả tối ưu (giống tham lam từ đầu). Tổng `O(n log n)`, **không có sai số nhị phân** (chỉ sai số phép chia số thực).

In [12]:
def proportional_heap(stations, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    L = sum(gaps)
    p = [1 + g * k // L for g in gaps]          # số nguyên: tránh sai số làm tròn
    r = k - sum(c - 1 for c in p)
    heap = [(-g / c, i) for i, (g, c) in enumerate(zip(gaps, p))]
    heapq.heapify(heap)
    for _ in range(r):
        _, i = heapq.heappop(heap)
        p[i] += 1
        heapq.heappush(heap, (-gaps[i] / p[i], i))
    return -heap[0][0]

for st, k, exp in EXAMPLES:
    print(proportional_heap(st, k), exp)

0.5 0.5
14.0 14.0


Chú ý: `g * k // L` dùng **phép chia nguyên** trên số nguyên Python (không tràn số) → `⌊g k / L⌋` chính xác, không bị lỗi làm tròn số thực.

**Code nộp LeetCode (Cách 4).**

In [13]:
class SolutionProportional:
    def minmaxGasDist(self, stations: List[int], k: int) -> float:
        import heapq
        gaps = [b - a for a, b in zip(stations, stations[1:])]
        L = sum(gaps)
        cnt = [1 + g * k // L for g in gaps]
        remaining = k - sum(c - 1 for c in cnt)
        heap = [(-g / c, i) for i, (g, c) in enumerate(zip(gaps, cnt))]
        heapq.heapify(heap)
        for _ in range(remaining):
            _, i = heapq.heappop(heap)
            cnt[i] += 1
            heapq.heappush(heap, (-gaps[i] / cnt[i], i))
        return -heap[0][0]

print([SolutionProportional().minmaxGasDist(st, k) for st, k, _ in EXAMPLES])

[0.5, 14.0]


**Độ phức tạp.** `O(n log n)` (heapify `O(n)` + tối đa `n` thao tác heap). Bộ nhớ `O(n)`.

### 7.4. Cách 5 — Nhị phân "chính xác" trên tập ứng viên `g_i / p`

**Giải thích (nâng cao).** Đáp án tối ưu luôn có dạng `g_i / p` với một gap `i` và một số nguyên `p` (đoạn dài nhất là một đoạn chia đều của gap nào đó). Sau khi nhị phân số thực được `hi ≈ D*` (luôn làm được), ta **chốt** đáp án chính xác:

1. Với mỗi gap, lấy số đoạn nhỏ nhất để mọi đoạn `≤ hi`: `p_i = ⌈g_i / hi⌉`. Phân bổ này dùng `need(hi) ≤ k` trạm.
2. Penalty thật của phân bổ đó là `max_i g_i / p_i` — một phân số đúng, nằm trong `[D*, hi]`. Vì các ứng viên `g/p` rời rạc và `hi − D*` cực nhỏ, giá trị này chính là `D*`.

Dùng `fractions.Fraction` để biểu diễn kết quả chính xác tuyệt đối (một trừ nhỏ `1e-12` bên trong `ceil` để chống lỗi làm tròn số thực khi `g/hi` "gần như nguyên").

In [14]:
from fractions import Fraction

def exact_answer(stations, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    D = binary_search(stations, k, iters=200)
    # p_i = số đoạn nhỏ nhất để g_i / p_i <= D
    p = [max(1, math.ceil(g / D - 1e-12)) for g in gaps]
    assert sum(c - 1 for c in p) <= k
    return max(Fraction(g, c) for g, c in zip(gaps, p))

for st, k, exp in EXAMPLES:
    ans = exact_answer(st, k)
    print(ans, float(ans), exp)

1/2 0.5 0.5
14 14.0 14.0


**Code nộp LeetCode (Cách 5)** — trả về `float` của phân số tìm được.

In [15]:
class SolutionExact:
    def minmaxGasDist(self, stations: List[int], k: int) -> float:
        import math
        gaps = [b - a for a, b in zip(stations, stations[1:])]
        lo, hi = 0.0, float(max(gaps))
        for _ in range(100):
            mid = (lo + hi) / 2
            if sum(math.ceil(g / mid) - 1 for g in gaps) <= k:
                hi = mid
            else:
                lo = mid
        p = [max(1, math.ceil(g / hi - 1e-12)) for g in gaps]
        return max(g / c for g, c in zip(gaps, p))

print([SolutionExact().minmaxGasDist(st, k) for st, k, _ in EXAMPLES])

[0.5, 14.0]


**Độ phức tạp.** Như Cách 3 cộng `O(n)` để chốt. Ưu điểm: kết quả là đúng một giá trị `g_i / p_i` (không "lơ lửng" trong khoảng sai số).

## 8. TRACE từng bước

Trace **Cách 3 (nhị phân)** trên ví dụ 2 (`k = 1`), 12 vòng đầu.

In [16]:
def trace_binary(stations, k, rounds=12):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    lo, hi = 0.0, float(max(gaps))
    print("gaps =", gaps)
    for it in range(rounds):
        mid = (lo + hi) / 2
        nd = sum(math.ceil(g / mid) - 1 for g in gaps)
        ok = nd <= k
        print(f"vòng {it + 1:2}: lo={lo:9.5f} hi={hi:9.5f} mid={mid:9.5f} need={nd:3} -> {'hi = mid' if ok else 'lo = mid'}")
        if ok:
            hi = mid
        else:
            lo = mid
    print("ước lượng sau", rounds, "vòng:", hi)

trace_binary([23, 24, 36, 39, 46, 56, 57, 65, 84, 98], 1)

gaps = [1, 12, 3, 7, 10, 1, 8, 19, 14]
vòng  1: lo=  0.00000 hi= 19.00000 mid=  9.50000 need=  4 -> lo = mid
vòng  2: lo=  9.50000 hi= 19.00000 mid= 14.25000 need=  1 -> hi = mid
vòng  3: lo=  9.50000 hi= 14.25000 mid= 11.87500 need=  3 -> lo = mid
vòng  4: lo= 11.87500 hi= 14.25000 mid= 13.06250 need=  2 -> lo = mid
vòng  5: lo= 13.06250 hi= 14.25000 mid= 13.65625 need=  2 -> lo = mid
vòng  6: lo= 13.65625 hi= 14.25000 mid= 13.95312 need=  2 -> lo = mid
vòng  7: lo= 13.95312 hi= 14.25000 mid= 14.10156 need=  1 -> hi = mid
vòng  8: lo= 13.95312 hi= 14.10156 mid= 14.02734 need=  1 -> hi = mid
vòng  9: lo= 13.95312 hi= 14.02734 mid= 13.99023 need=  2 -> lo = mid
vòng 10: lo= 13.99023 hi= 14.02734 mid= 14.00879 need=  1 -> hi = mid
vòng 11: lo= 13.99023 hi= 14.00879 mid= 13.99951 need=  2 -> lo = mid
vòng 12: lo= 13.99951 hi= 14.00879 mid= 14.00415 need=  1 -> hi = mid
ước lượng sau 12 vòng: 14.004150390625


Bảng tóm tắt vài vòng đầu:

| vòng | lo | hi | mid | need(mid) | quyết định |
|---|---|---|---|---|---|
| 1 | 0 | 19 | 9.5 | 7 | lo = 9.5 |
| 2 | 9.5 | 19 | 14.25 | 1 | hi = 14.25 |
| 3 | 9.5 | 14.25 | 11.875 | 4 | lo = 11.875 |
| ... | | | | | hội tụ về 14 |

Trace **Cách 2 (heap)** trên gaps `[10, 6]` (`stations = [0, 10, 16]`), `k = 3`:

In [17]:
def trace_heap(stations, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    p = [1] * len(gaps)
    heap = [(-g, i) for i, g in enumerate(gaps)]
    heapq.heapify(heap)
    for step in range(1, k + 1):
        neg, i = heapq.heappop(heap)
        p[i] += 1
        heapq.heappush(heap, (-gaps[i] / p[i], i))
        print(f"trạm {step}: chia gap #{i} (đang dài {-neg:.3f}) -> p = {p}, "
              f"các đoạn = {[round(g / c, 3) for g, c in zip(gaps, p)]}")
    print("penalty =", -heap[0][0])

trace_heap([0, 10, 16], 3)

trạm 1: chia gap #0 (đang dài 10.000) -> p = [2, 1], các đoạn = [5.0, 6.0]
trạm 2: chia gap #1 (đang dài 6.000) -> p = [2, 2], các đoạn = [5.0, 3.0]
trạm 3: chia gap #0 (đang dài 5.000) -> p = [3, 2], các đoạn = [3.333, 3.0]
penalty = 3.3333333333333335


Trace **Cách 4**: phân bổ tỉ lệ ban đầu trên ví dụ 1 và một ví dụ lớn hơn.

In [18]:
def trace_proportional(stations, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    L = sum(gaps)
    p = [1 + g * k // L for g in gaps]
    r = k - sum(c - 1 for c in p)
    print(f"L = {L}, k = {k}, phân bổ ban đầu p = {p}, còn lại r = {r} (< n = {len(gaps)})")
    print("đoạn lớn nhất ban đầu:", max(g / c for g, c in zip(gaps, p)), "< L/k =", L / k)
    print("đáp án:", proportional_heap(stations, k))

trace_proportional(list(range(1, 11)), 9)
trace_proportional([23, 24, 36, 39, 46, 56, 57, 65, 84, 98], 25)

L = 9, k = 9, phân bổ ban đầu p = [2, 2, 2, 2, 2, 2, 2, 2, 2], còn lại r = 0 (< n = 9)
đoạn lớn nhất ban đầu: 0.5 < L/k = 1.0
đáp án: 0.5
L = 75, k = 25, phân bổ ban đầu p = [1, 5, 2, 3, 4, 1, 3, 7, 5], còn lại r = 3 (< n = 9)
đoạn lớn nhất ban đầu: 2.8 < L/k = 3.0
đáp án: 2.5


## 9. Code nộp LeetCode

Bản chọn: **Cách 3 (nhị phân trên số thực)** — ngắn, chuẩn mực, `O(n log(W/ε))`. (Cách 4 là lựa chọn thay thế chính xác và cũng nhanh.)

In [19]:
class Solution:
    def minmaxGasDist(self, stations: List[int], k: int) -> float:
        gaps = [b - a for a, b in zip(stations, stations[1:])]
        lo, hi = 0.0, float(max(gaps))      # hi luôn "làm được" (0 trạm)
        while hi - lo > 1e-6:
            mid = (lo + hi) / 2
            # số trạm cần thêm để mọi đoạn <= mid
            if sum(int(g / mid) for g in gaps) <= k:
                hi = mid                    # làm được -> thử nhỏ hơn
            else:
                lo = mid                    # không đủ trạm -> phải nới D
        return hi

for st, k, exp in EXAMPLES:
    assert close(Solution().minmaxGasDist(st, k), exp)
print("OK")

OK


Giải thích từng dòng:

- `gaps`: chỉ khoảng trống quan trọng.
- `hi = max(gaps)`: không thêm trạm nào thì penalty là gap lớn nhất — chắc chắn làm được.
- `lo = 0`: không bao giờ làm được (cần vô hạn trạm).
- `int(g / mid)`: số trạm để chia `g` thành các đoạn `≤ mid` (sai khác `⌈g/mid⌉ − 1` đúng 1 khi `g/mid` nguyên — vô hại khi nhị phân số thực).
- Điều kiện dừng `hi − lo > 1e-6`: đáp án nằm trong `[lo, hi]`, trả `hi` sai số `≤ 1e-6`.

## 10. Vì sao thuật toán đúng

**Bổ đề 1 (chia đều tối ưu).** Với `p` đoạn trong gap `g`, đoạn lớn nhất `≥ g/p`, đạt dấu bằng khi chia đều. Do đó penalty của phân bổ `(p_i)` là `max g_i/p_i`, và `need(D) = Σ (⌈g_i/D⌉ − 1)` là số trạm **tối thiểu** để penalty `≤ D`.

**Bổ đề 2 (đơn điệu).** `need(D)` không tăng theo `D`. Vậy `{D : need(D) ≤ k}` là một khoảng `[D*, ∞)` → nhị phân hội tụ về `D*`; bất biến `need(hi) ≤ k < need(lo)`... (với `lo` = 0 ta quy ước `need(0) = ∞`).

**Tham lam (Cách 1, 2, 4).** Gọi `D*` là đáp án tối ưu và `q_i = ⌈g_i/D*⌉` là phân bổ tối ưu "tối thiểu" (`Σ(q_i − 1) ≤ k`). Bất biến: trong quá trình tham lam, `p_i ≤ q_i` với mọi `i`, cho tới khi mọi đoạn `≤ D*`. Thật vậy, mỗi lần tham lam chọn gap có `g_i/p_i` lớn nhất; nếu giá trị này `> D*` thì `p_i < q_i` (vì `g_i/q_i ≤ D*`), nên tăng `p_i` vẫn giữ `p_i ≤ q_i`. Vì `Σ(q_i − 1) ≤ k`, sau tối đa `k` bước mọi đoạn `≤ D*`; các bước còn lại chỉ làm penalty giảm hoặc giữ nguyên → kết quả `≤ D*`, tức bằng `D*`. Cách 4 bắt đầu từ một phân bổ thoả `p_i ≤ q_i` (đã chứng minh ở 7.3) nên cùng lập luận áp dụng.

## 11. Phân tích Time/Space

Gọi `n` = số gap, `W` = gap lớn nhất (≤ `10^8`), `ε` = `10^-6`.

| Cách | Time | Space |
|---|---|---|
| 1. Tham lam quét | `O(k · n)` | `O(n)` |
| 2. Tham lam heap | `O(n + k log n)` | `O(n)` |
| 3. Nhị phân số thực | `O(n log(W/ε))` ≈ `47n` | `O(n)` |
| 4. Tỉ lệ + heap | `O(n log n)` | `O(n)` |
| 5. Nhị phân + chốt phân số | `O(n log(W/ε))` | `O(n)` |

In [20]:
import random, time
random.seed(774)
st_big = sorted(random.sample(range(0, 10 ** 8), 2000))
for k in [1000, 100000]:
    for name, f in [("heap", greedy_heap), ("binary", binary_search),
                    ("proportional", proportional_heap), ("Solution", lambda s, kk: Solution().minmaxGasDist(s, kk))]:
        t0 = time.perf_counter()
        r = f(st_big, k)
        print(f"k={k:>6} {name:12} {r:.7f}  {1000 * (time.perf_counter() - t0):8.1f} ms")
t0 = time.perf_counter()
r = proportional_heap(st_big, 10 ** 6)
print(f"k=1000000 proportional {r:.7f}  {1000 * (time.perf_counter() - t0):8.1f} ms")

k=  1000 heap         54534.0000000       1.4 ms
k=  1000 binary       54534.0000000      12.2 ms
k=  1000 proportional 54534.0000000       0.9 ms
k=  1000 Solution     54534.0000000       4.0 ms
k=100000 heap         988.3039216      53.1 ms


k=100000 binary       988.3039216      12.8 ms
k=100000 proportional 988.3039216       1.2 ms
k=100000 Solution     988.3039220       4.1 ms
k=1000000 proportional 99.7148438       1.3 ms


## 12. Bộ test

In [21]:
methods = {
    "scan": greedy_scan, "heap": greedy_heap, "binary": binary_search,
    "proportional": proportional_heap,
    "exact": lambda s, k: float(exact_answer(s, k)),
    "SolutionHeap": lambda s, k: SolutionHeap().minmaxGasDist(s, k),
    "SolutionBinary": lambda s, k: SolutionBinary().minmaxGasDist(s, k),
    "SolutionProportional": lambda s, k: SolutionProportional().minmaxGasDist(s, k),
    "SolutionExact": lambda s, k: SolutionExact().minmaxGasDist(s, k),
    "Solution": lambda s, k: Solution().minmaxGasDist(s, k),
}

# 12.1 Ví dụ
for name, f in methods.items():
    for st, k, exp in EXAMPLES:
        assert close(f(st, k), exp), (name, st, k, f(st, k))

# 12.2 Edge cases
edge = [
    ([0, 10], 1, 5.0),
    ([0, 10], 4, 2.0),
    ([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 1, 1.0),          # mọi gap = 1: thêm 1 trạm không giảm được max
    ([0, 3, 4], 2, 1.0),                                # gap 3 chia 3 -> 1
    ([0, 10, 16], 3, 10 / 3),
    ([0, 100000000], 1000000, 100000000 / 1000001),
]
for name, f in methods.items():
    for st, k, exp in edge:
        if name == "scan" and k > 10 ** 4:
            continue
        assert close(f(st, k), exp), (name, st, k, f(st, k), exp)
print("Ví dụ + edge cases: OK")

Ví dụ + edge cases: OK


In [22]:
# 12.3 Ngẫu nhiên: đối chiếu với "brute force" quy hoạch động chính xác trên phân số (n, k nhỏ)
from functools import lru_cache

def brute_exact(stations, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    @lru_cache(maxsize=None)
    def best(i, rem):
        # penalty nhỏ nhất cho gaps[i:] dùng đúng/tối đa rem trạm
        if i == len(gaps):
            return Fraction(0)
        res = None
        for extra in range(rem + 1):
            v = max(Fraction(gaps[i], extra + 1), best(i + 1, rem - extra))
            res = v if res is None or v < res else res
        return res
    return best(0, k)

random.seed(1)
for _ in range(300):
    n = random.randint(2, 6)
    st = sorted(random.sample(range(0, 40), n))
    k = random.randint(1, 8)
    ref = float(brute_exact(st, k))
    for name, f in methods.items():
        assert close(f(st, k), ref), (name, st, k, f(st, k), ref)
    assert exact_answer(st, k) == brute_exact(st, k)
print("Đối chiếu 300 test ngẫu nhiên với DP phân số: OK")

# 12.4 Ngẫu nhiên lớn: các cách nhanh khớp nhau
for _ in range(30):
    st = sorted(random.sample(range(0, 10 ** 8), random.randint(10, 2000)))
    k = random.randint(1, 20000)
    vals = [greedy_heap(st, k), binary_search(st, k), proportional_heap(st, k), Solution().minmaxGasDist(st, k)]
    assert max(vals) - min(vals) <= 1e-6, vals
print("Ngẫu nhiên lớn: OK")

Đối chiếu 300 test ngẫu nhiên với DP phân số: OK


Ngẫu nhiên lớn: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tham lam "chia đôi khoảng dài nhất" (đặt trạm vào giữa đoạn hiện tại)

**Code sai:**

In [23]:
def wrong_halving(stations, k):
    segs = [-(b - a) for a, b in zip(stations, stations[1:])]
    heapq.heapify(segs)
    for _ in range(k):
        g = -heapq.heappop(segs)
        heapq.heappush(segs, -g / 2)              # SAI: chia đôi đoạn hiện tại
        heapq.heappush(segs, -g / 2)
    return -segs[0]

st, k = [0, 10, 16], 3
print("sai:", wrong_halving(st, k), " đúng:", greedy_heap(st, k))
print("sai:", wrong_halving([0, 9], 2), " đúng:", greedy_heap([0, 9], 2))

sai: 5.0  đúng: 3.3333333333333335
sai: 4.5  đúng: 3.0


**Vì sao sai:** chia đôi liên tiếp chỉ tạo ra các đoạn `g/2, g/4, ...` — không thể có `g/3`. Ví dụ gap 9 với 2 trạm: tối ưu là `9/3 = 3`, còn chia đôi cho `max(4.5/2, 4.5) = 4.5`.

**Code đúng:** đếm số đoạn `p_i` của mỗi gap ban đầu và luôn **chia đều lại** `g_i / p_i` (Cách 2).

### Lỗi 2 — Nhị phân trên số nguyên

**Code sai:**

In [24]:
def wrong_integer_bs(stations, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    lo, hi = 1, max(gaps)
    while lo < hi:                                # SAI: đáp án là số thực
        mid = (lo + hi) // 2
        if sum(math.ceil(g / mid) - 1 for g in gaps) <= k:
            hi = mid
        else:
            lo = mid + 1
    return float(lo)

print("sai:", wrong_integer_bs(list(range(1, 11)), 9), " đúng:", Solution().minmaxGasDist(list(range(1, 11)), 9))

sai: 1.0  đúng: 0.5000009536743164


**Vì sao sai:** đáp án `0.5` không phải số nguyên. **Code đúng:** nhị phân trên `float`, dừng theo `eps` hoặc số vòng cố định.

### Lỗi 3 — Công thức số trạm cần sai: `g / D` (không làm tròn) hoặc `⌈g / D⌉`

**Code sai:**

In [25]:
def wrong_need(stations, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    lo, hi = 0.0, float(max(gaps))
    for _ in range(100):
        mid = (lo + hi) / 2
        if sum(math.ceil(g / mid) for g in gaps) <= k:    # SAI: đếm số ĐOẠN, không phải số TRẠM
            hi = mid
        else:
            lo = mid
    return hi

for st, k, exp in EXAMPLES:
    print("sai:", round(wrong_need(st, k), 5), " đúng:", exp)

sai: 1.0  đúng: 0.5
sai: 19.0  đúng: 14.0


**Vì sao sai:** `⌈g/D⌉` là số **đoạn**; số **trạm** cần thêm là số đoạn trừ 1. Đếm thừa `n` trạm làm đáp án quá lớn (thậm chí không bao giờ thoả nếu `k < n`, trả về `max(gaps)`).

**Code đúng:** `⌈g/D⌉ − 1` (hoặc `int(g/D)`).

### Lỗi 4 — Dùng `while lo < hi` với số thực (vòng lặp vô hạn) hoặc eps quá lớn

In [26]:
def demo_eps(stations, k, eps):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    lo, hi, it = 0.0, float(max(gaps)), 0
    while hi - lo > eps:
        it += 1
        mid = (lo + hi) / 2
        if sum(int(g / mid) for g in gaps) <= k:
            hi = mid
        else:
            lo = mid
    return hi, it

st, k = [0, 100000000], 1000000
print("eps=1e-2:", demo_eps(st, k, 1e-2), "| eps=1e-6:", demo_eps(st, k, 1e-6), "| đúng:", 1e8 / (k + 1))

eps=1e-2: (100.000761449337, 34) | eps=1e-6: (99.99990027154126, 47) | đúng: 99.9999000001


**Vì sao sai:** `while lo < hi` với `float` có thể không bao giờ dừng (khi `mid` bằng `lo` hoặc `hi` do giới hạn độ chính xác, vòng lặp đứng yên). Eps quá lớn (`1e-2`) cho đáp án lệch quá `1e-6`.

**Code đúng:** `while hi - lo > 1e-6` hoặc `for _ in range(100)`.

## 14. Mẹo & nhận dạng pattern

- **"Cực tiểu hoá giá trị lớn nhất"** / **"cực đại hoá giá trị nhỏ nhất"** → tìm kiếm nhị phân trên đáp án + hàm kiểm tra tham lam (*410. Split Array Largest Sum*, *875. Koko Eating Bananas*, *1011. Capacity To Ship Packages*, *1552. Magnetic Force Between Two Balls*).
- **Đáp án số thực** → nhị phân với `eps` hoặc số vòng cố định; không dùng `lo < hi`.
- **Phân bổ `k` đơn vị cho `n` nhóm, mỗi lần đơn vị cho nhóm "tệ nhất"** → heap tham lam; nếu `k` lớn, **phân bổ thô theo tỉ lệ trước** rồi heap phần dư.
- **Chia khoảng thành `p` phần** → chia đều là tối ưu; số điểm cần `⌈g/D⌉ − 1`.

## 15. Biến thể

1. **Vị trí trạm mới phải là số nguyên** → nhị phân trên số nguyên với `need(D) = Σ (⌈g/D⌉ − 1)`.
2. **Cực đại hoá khoảng cách nhỏ nhất khi chọn `m` điểm từ tập cho trước** (*1552*) → nhị phân + tham lam.
3. **Đường tròn** (trạm trên vòng tròn chu vi `C`) → thêm gap `C − last + first`.
4. **Trả về vị trí các trạm mới** → sau khi có `p_i`, đặt các trạm chia đều mỗi gap.
5. **Mỗi trạm có chi phí khác nhau theo vị trí** → không còn đơn giản; cần DP.

In [27]:
# Biến thể 4: vị trí các trạm mới (dùng phân bổ của Cách 4)
def new_station_positions(stations, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    L = sum(gaps)
    p = [1 + g * k // L for g in gaps]
    heap = [(-g / c, i) for i, (g, c) in enumerate(zip(gaps, p))]
    heapq.heapify(heap)
    for _ in range(k - sum(c - 1 for c in p)):
        _, i = heapq.heappop(heap)
        p[i] += 1
        heapq.heappush(heap, (-gaps[i] / p[i], i))
    pos = []
    for a, g, c in zip(stations, gaps, p):
        pos += [a + g * t / c for t in range(1, c)]
    return pos

pos = new_station_positions([0, 10, 16], 3)
allp = sorted([0, 10, 16] + pos)
print(pos, "max gap =", max(b - a for a, b in zip(allp, allp[1:])))

# Biến thể 1: trạm mới tại vị trí nguyên
def integer_positions(stations, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    lo, hi = 1, max(gaps)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((g + mid - 1) // mid - 1 for g in gaps) <= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

print(integer_positions(list(range(1, 11)), 9), integer_positions([0, 10, 16], 3))

[3.3333333333333335, 6.666666666666667, 13.0] max gap = 3.3333333333333335
1 4


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tham lam quét | Mỗi trạm vào gap có đoạn dài nhất | `O(k·n)` | `O(n)` | Minh hoạ, `k` nhỏ |
| 2. Tham lam heap | Như 1, dùng heap | `O(k log n)` | `O(n)` | `k` vừa phải |
| 3. Nhị phân số thực | `need(D) <= k` | `O(n log(W/ε))` | `O(n)` | Mặc định, nộp bài |
| 4. Tỉ lệ + heap | Phân bổ thô theo tỉ lệ, heap phần dư `< n` | `O(n log n)` | `O(n)` | Chính xác, nhanh |
| 5. Nhị phân + chốt | Nhị phân rồi chốt `g_i/p_i` | `O(n log(W/ε))` | `O(n)` | Cần giá trị phân số đúng |

## 17. Bài tập

**Bài 1 (đoán output).** `minmaxGasDist([0, 7, 11], 2)` = ? và với `k = 3`?

**Bài 2 (tính tay).** Với gaps `[12, 5, 8]` và `D = 4`, tính `need(D)`. Với `k = 4`, đáp án `≤ 4` hay `> 4`?

**Bài 3 (điền TODO).** Hoàn thiện `need_stations(gaps, D)` trả về số trạm tối thiểu để mọi đoạn `≤ D`.

**Bài 4 (tự viết lại).** Viết lại Cách 3 dùng **đúng 100 vòng lặp** và trả về `(lo + hi) / 2`.

**Bài 5 (mini challenge).** Trạm nằm trên **đường tròn** chu vi `C` (vị trí trong `[0, C)`). Viết `circular_minmax(stations, C, k)` và kiểm tra bằng DP phân số.

In [28]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def need_stations(gaps, D):
    # TODO: sum(ceil(g / D) - 1)
    pass

# Bài 4 — TODO
def binary_fixed_iters(stations, k):
    # TODO: for _ in range(100): ...
    pass

# Bài 5 — TODO
def circular_minmax(stations, C, k):
    # TODO: thêm gap C - stations[-1] + stations[0], rồi nhị phân
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: gaps `[7, 4]`.
- Mức 2: `k = 2`: thử chia 7 thành 3 phần → `2.33`, gap 4 còn `4` → max 4; hoặc 7 thành 2 (3.5) và 4 thành 2 (2) → max 3.5.
- Mức 3: `k = 2` → `3.5`; `k = 3` → `max(7/3, 4/2) = 2.333...`.

**Bài 2.**
- Mức 1: `⌈12/4⌉ − 1 = 2`, `⌈5/4⌉ − 1 = 1`, `⌈8/4⌉ − 1 = 1`.
- Mức 2: `need(4) = 4`.
- Mức 3: `4 <= k = 4` → làm được với `D = 4` → đáp án `≤ 4`.

**Bài 3.**
- Mức 1: duyệt từng gap.
- Mức 2: dùng `math.ceil`.
- Mức 3: trừ 1 cho mỗi gap.

**Bài 4.**
- Mức 1: thay `while` bằng `for _ in range(100)`.
- Mức 2: giữ bất biến `need(hi) <= k`.
- Mức 3: trả `(lo + hi) / 2` hoặc `hi`.

**Bài 5.**
- Mức 1: đường tròn có `n` gap (thêm gap nối cuối với đầu).
- Mức 2: gap vòng = `C − stations[-1] + stations[0]`.
- Mức 3: phần còn lại giống hệt Cách 3.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
assert close(Solution().minmaxGasDist([0, 7, 11], 2), 3.5)
assert close(Solution().minmaxGasDist([0, 7, 11], 3), 7 / 3)

# Bài 2
def need_stations_ans(gaps, D):
    return sum(math.ceil(g / D) - 1 for g in gaps)
assert need_stations_ans([12, 5, 8], 4) == 4

# Bài 4
def binary_fixed_iters_ans(stations, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])]
    lo, hi = 0.0, float(max(gaps))
    for _ in range(100):
        mid = (lo + hi) / 2
        if need_stations_ans(gaps, mid) <= k:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2

# Bài 5
def circular_minmax_ans(stations, C, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])] + [C - stations[-1] + stations[0]]
    lo, hi = 0.0, float(max(gaps))
    for _ in range(100):
        mid = (lo + hi) / 2
        if need_stations_ans(gaps, mid) <= k:
            hi = mid
        else:
            lo = mid
    return hi

def brute_circular(stations, C, k):
    gaps = [b - a for a, b in zip(stations, stations[1:])] + [C - stations[-1] + stations[0]]
    @lru_cache(maxsize=None)
    def best(i, rem):
        if i == len(gaps):
            return Fraction(0)
        return min(max(Fraction(gaps[i], e + 1), best(i + 1, rem - e)) for e in range(rem + 1))
    return best(0, k)

random.seed(55)
for _ in range(200):
    C = random.randint(5, 40)
    st = sorted(random.sample(range(C), random.randint(2, min(5, C))))
    k = random.randint(1, 6)
    assert close(circular_minmax_ans(st, C, k), float(brute_circular(st, C, k)))
    lin = sorted(random.sample(range(0, 40), random.randint(2, 6)))
    assert close(binary_fixed_iters_ans(lin, k), float(brute_exact(lin, k)))
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template nhị phân trên số thực:**

```python
gaps = [b - a for a, b in zip(stations, stations[1:])]
lo, hi = 0.0, float(max(gaps))
while hi - lo > 1e-6:                       # hoặc for _ in range(100)
    mid = (lo + hi) / 2
    if sum(int(g / mid) for g in gaps) <= k:   # đủ trạm để mọi đoạn <= mid
        hi = mid
    else:
        lo = mid
return hi
```

**Template tỉ lệ + heap:** `p_i = 1 + g_i*k//L`; heap `(-g_i/p_i, i)` cho `r = k − Σ(p_i − 1)` trạm còn lại.

**Checklist trước khi submit:**

- [ ] Làm việc trên **gap**, không trên vị trí.
- [ ] Số trạm cần cho gap `g` với ngưỡng `D` là `⌈g/D⌉ − 1` (≈ `int(g/D)`), không phải `⌈g/D⌉`.
- [ ] Nhị phân trên **số thực**; dừng bằng `eps` hoặc số vòng cố định.
- [ ] `hi = max(gaps)` (luôn làm được), `lo = 0`.
- [ ] Tham lam phải **chia đều lại** gap gốc (`g_i / p_i`), không chia đôi đoạn hiện tại.
- [ ] Với `k` tới `10^6`, tránh heap thuần trong Python (hoặc phân bổ tỉ lệ trước).
- [ ] Tên method đúng: `minmaxGasDist(self, stations, k)`.